# Dataframe erweitern mit dem korrekten Baujahr einer Adresse

In [38]:
# Imports
import pandas as pd
import requests
from typing import Any, Dict
from pathlib import Path

# Einstellungen für die Darstellung
data_dir = Path("Data")
data_dir.mkdir(exist_ok=True)

In [39]:
# Excel-Datei aus Notebook 01 einlesen
df = pd.read_excel(data_dir / "01_Datenanalyse_Ergebnis.xlsx").copy()

print("Shape:", df.shape)
display(df.head())

Shape: (846, 55)


,EGID,GEB_GEBID,GSW_STATUS,STRASSENNAME,HAUSNR,HAUSNRZUSATZ,PLZ4,ORT,STADTKREIS,HAUPTNUTZUNG,...,Fläche,Speziell,Fläche10,Holz,Holz Lm,Fläche11,Extra,Unnamed: 52,Unnamed: 53,Unnamed: 54
0,210294075,35999,bestehend,Albert-Einstein-Strasse,1,0,8404,Winterthur,Oberwinterthur,Industrie und Gerwerbe,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,191993323,36666,bestehend,Schützenwiesenweg,8,0,8400,Winterthur,Winterthur-Stadt,Verwaltungsgebäude und Gebäude mit öffentliche...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,191975144,34926,im Bau,Sporrerpark,2,0,8408,Winterthur,Wülflingen,Nebengebäude und dev.Gebäude,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,191975148,34924,im Bau,Sporrerpark,5,0,8408,Winterthur,Wülflingen,Wohngebäude,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,191975148,34924,im Bau,Sporrerpark,4,0,8408,Winterthur,Wülflingen,Wohngebäude,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [40]:
BASE = "https://api3.geo.admin.ch/rest/services/ech"

session = requests.Session()
session.headers.update({"User-Agent": "Bachelorarbeit-WWR/1.0"})

In [41]:
def build_address_from_row(row: pd.Series) -> str:
    strasse = str(row["STRASSENNAME"]).strip()
    hausnr = str(row["HAUSNR"]).strip()

    zusatz = row.get("HAUSNRZUSATZ", "")
    if pd.notna(zusatz):
        zusatz = str(zusatz).strip()
        if zusatz not in ("", "0", "nan", "None"):
            hausnr = f"{hausnr}{zusatz}"

    plz = str(row["PLZ4"]).strip()
    ort = str(row["ORT"]).strip()

    return f"{strasse} {hausnr}, {plz} {ort}"

In [42]:
# SearchServer-Funktion
def geocode_address(address: str) -> Dict[str, Any]:
    url = f"{BASE}/SearchServer"
    params = {
        "searchText": address,
        "type": "locations",
        "origins": "address",
        "sr": 2056
    }

    r = session.get(url, params=params, timeout=30)
    r.raise_for_status()
    data = r.json()

    results = data.get("results", [])
    if not results:
        raise ValueError(f"Keine Treffer für Adresse: {address}")

    return results[0]

In [43]:
# GWR-Feature laden
def get_gwr_feature(feature_id: str) -> Dict[str, Any]:
    layer = "ch.bfs.gebaeude_wohnungs_register"
    url = f"{BASE}/MapServer/{layer}/{feature_id}"
    params = {
        "returnGeometry": "false",
        "sr": 2056
    }

    r = session.get(url, params=params, timeout=30)
    r.raise_for_status()
    data = r.json()

    feature = data.get("feature")
    if not feature:
        raise ValueError(f"Kein GWR-Feature gefunden für feature_id={feature_id}")

    return feature

In [44]:
# Baujahr einer Adresse holen
def get_baujahr_for_address(address: str) -> Dict[str, Any]:
    search_hit = geocode_address(address)
    attrs = search_hit["attrs"]
    feature_id = attrs["featureId"]

    feature = get_gwr_feature(feature_id)
    gwr_attrs = feature.get("attributes", {})

    return {
        "adresse_api": address,
        "feature_id": feature_id,
        "egid_gwr": gwr_attrs.get("egid"),
        "baujahr_gwr": gwr_attrs.get("gbauj"),
        "label_api": attrs.get("label"),
        "x_api": attrs.get("x"),
        "y_api": attrs.get("y"),
    }

In [45]:
# Einzeltest mit erster Zeile
row0 = df.iloc[0]
test_address = build_address_from_row(row0)

test_result = get_baujahr_for_address(test_address)
print(test_result)

{'adresse_api': 'Albert-Einstein-Strasse 1, 8404 Winterthur', 'feature_id': '210294075_0', 'egid_gwr': '210294075', 'baujahr_gwr': 2024, 'label_api': 'Albert-Einstein-Strasse 1 <b>8404 Winterthur</b>', 'x_api': 1263647.75, 'y_api': 2700009.75}


In [46]:
# API-Abfrage für alle Adressen aus df
resultate = []

for idx, row in df.iterrows():
    address = build_address_from_row(row)

    try:
        res = get_baujahr_for_address(address)

        resultate.append({
            "row_index": idx,
            "adresse_api": res["adresse_api"],
            "feature_id": res["feature_id"],
            "egid_gwr": res["egid_gwr"],
            "baujahr_gwr": res["baujahr_gwr"],
            "label_api": res["label_api"],
            "x_api": res["x_api"],
            "y_api": res["y_api"],
            "status_api": "OK"
        })

        print(f"[OK] {idx}: {address} -> Baujahr {res['baujahr_gwr']}")

    except Exception as e:
        resultate.append({
            "row_index": idx,
            "adresse_api": address,
            "feature_id": None,
            "egid_gwr": None,
            "baujahr_gwr": None,
            "label_api": None,
            "x_api": None,
            "y_api": None,
            "status_api": f"FEHLER: {e}"
        })

        print(f"[FEHLER] {idx}: {address} -> {e}")

df_baujahr_api = pd.DataFrame(resultate)
display(df_baujahr_api.head())

[OK] 0: Albert-Einstein-Strasse 1, 8404 Winterthur -> Baujahr 2024
[OK] 1: Schützenwiesenweg 8, 8400 Winterthur -> Baujahr 2024
[OK] 2: Sporrerpark 2, 8408 Winterthur -> Baujahr 2024
[OK] 3: Sporrerpark 5, 8408 Winterthur -> Baujahr 2024
[OK] 4: Sporrerpark 4, 8408 Winterthur -> Baujahr 2024
[OK] 5: St. Gallerstrasse 133, 8404 Winterthur -> Baujahr 2024
[OK] 6: Grüzefeldstrasse 34, 8400 Winterthur -> Baujahr 2023
[OK] 7: Guggenbühlstrasse 140a, 8404 Winterthur -> Baujahr 2023
[OK] 8: Hörnlistrasse 31, 8400 Winterthur -> Baujahr 2023
[OK] 9: Im Hölderli 3, 8405 Winterthur -> Baujahr 2023
[OK] 10: Kiesstrasse 4, 8400 Winterthur -> Baujahr 2023
[OK] 11: Rennweg 5a, 8400 Winterthur -> Baujahr 2023
[OK] 12: Rennweg 5a, 8400 Winterthur -> Baujahr 2023
[OK] 13: Schützenstrasse 15a, 8400 Winterthur -> Baujahr 2023
[OK] 14: Talhofweg 24b, 8408 Winterthur -> Baujahr 2023
[OK] 15: Wurmbühlstrasse 9d, 8405 Winterthur -> Baujahr 2023
[OK] 16: Obermühlestrasse 5, 8400 Winterthur -> Baujahr 2022
[OK]

,row_index,adresse_api,feature_id,egid_gwr,baujahr_gwr,label_api,x_api,y_api,status_api
0,0,"Albert-Einstein-Strasse 1, 8404 Winterthur",210294075_0,210294075,2024.0,Albert-Einstein-Strasse 1 <b>8404 Winterthur</b>,1263647.750,2700009.75,OK
1,1,"Schützenwiesenweg 8, 8400 Winterthur",191993323_0,191993323,2024.0,Schützenwiesenweg 8 <b>8400 Winterthur</b>,1261888.125,2696149.25,OK
2,2,"Sporrerpark 2, 8408 Winterthur",191975144_0,191975144,2024.0,Sporrerpark 2 <b>8408 Winterthur</b>,1263772.625,2694387.25,OK
3,3,"Sporrerpark 5, 8408 Winterthur",191975148_1,191975148,2024.0,Sporrerpark 5 <b>8408 Winterthur</b>,1263791.000,2694430.75,OK
4,4,"Sporrerpark 4, 8408 Winterthur",191975148_0,191975148,2024.0,Sporrerpark 4 <b>8408 Winterthur</b>,1263792.375,2694416.25,OK


In [47]:
# API-Resultate zurück an dein ursprüngliches df hängen
df_result = df.reset_index(drop=True).copy()
df_result = pd.concat(
    [df_result, df_baujahr_api.drop(columns=["row_index"])],
    axis=1
)

display(df_result.head())

,EGID,GEB_GEBID,GSW_STATUS,STRASSENNAME,HAUSNR,HAUSNRZUSATZ,PLZ4,ORT,STADTKREIS,HAUPTNUTZUNG,...,Unnamed: 53,Unnamed: 54,adresse_api,feature_id,egid_gwr,baujahr_gwr,label_api,x_api,y_api,status_api
0,210294075,35999,bestehend,Albert-Einstein-Strasse,1,0,8404,Winterthur,Oberwinterthur,Industrie und Gerwerbe,...,NaN,NaN,"Albert-Einstein-Strasse 1, 8404 Winterthur",210294075_0,210294075,2024.0,Albert-Einstein-Strasse 1 <b>8404 Winterthur</b>,1263647.750,2700009.75,OK
1,191993323,36666,bestehend,Schützenwiesenweg,8,0,8400,Winterthur,Winterthur-Stadt,Verwaltungsgebäude und Gebäude mit öffentliche...,...,NaN,NaN,"Schützenwiesenweg 8, 8400 Winterthur",191993323_0,191993323,2024.0,Schützenwiesenweg 8 <b>8400 Winterthur</b>,1261888.125,2696149.25,OK
2,191975144,34926,im Bau,Sporrerpark,2,0,8408,Winterthur,Wülflingen,Nebengebäude und dev.Gebäude,...,NaN,NaN,"Sporrerpark 2, 8408 Winterthur",191975144_0,191975144,2024.0,Sporrerpark 2 <b>8408 Winterthur</b>,1263772.625,2694387.25,OK
3,191975148,34924,im Bau,Sporrerpark,5,0,8408,Winterthur,Wülflingen,Wohngebäude,...,NaN,NaN,"Sporrerpark 5, 8408 Winterthur",191975148_1,191975148,2024.0,Sporrerpark 5 <b>8408 Winterthur</b>,1263791.000,2694430.75,OK
4,191975148,34924,im Bau,Sporrerpark,4,0,8408,Winterthur,Wülflingen,Wohngebäude,...,NaN,NaN,"Sporrerpark 4, 8408 Winterthur",191975148_0,191975148,2024.0,Sporrerpark 4 <b>8408 Winterthur</b>,1263792.375,2694416.25,OK


In [48]:
# Vergleich mit ursprünglichem Baujahr aus df
if "BAUJAHR" in df_result.columns:
    df_result["BAUJAHR"] = pd.to_numeric(df_result["BAUJAHR"], errors="coerce")
    df_result["baujahr_gwr"] = pd.to_numeric(df_result["baujahr_gwr"], errors="coerce")

    df_result["baujahr_gleich"] = df_result["BAUJAHR"] == df_result["baujahr_gwr"]

    print("Vergleich lokale BAUJAHR-Spalte vs. API:")
    print(df_result["baujahr_gleich"].value_counts(dropna=False))

    display(
        df_result[[
            "adresse_api",
            "BAUJAHR",
            "baujahr_gwr",
            "baujahr_gleich",
            "status_api"
        ]].head(20)
    )

Vergleich lokale BAUJAHR-Spalte vs. API:
baujahr_gleich
True     814
False     32
Name: count, dtype: int64


,adresse_api,BAUJAHR,baujahr_gwr,baujahr_gleich,status_api
0,"Albert-Einstein-Strasse 1, 8404 Winterthur",2024,2024.0,True,OK
1,"Schützenwiesenweg 8, 8400 Winterthur",2024,2024.0,True,OK
2,"Sporrerpark 2, 8408 Winterthur",2024,2024.0,True,OK
3,"Sporrerpark 5, 8408 Winterthur",2024,2024.0,True,OK
4,"Sporrerpark 4, 8408 Winterthur",2024,2024.0,True,OK
5,"St. Gallerstrasse 133, 8404 Winterthur",2024,2024.0,True,OK
6,"Grüzefeldstrasse 34, 8400 Winterthur",2023,2023.0,True,OK
7,"Guggenbühlstrasse 140a, 8404 Winterthur",2023,2023.0,True,OK
8,"Hörnlistrasse 31, 8400 Winterthur",2023,2023.0,True,OK
9,"Im Hölderli 3, 8405 Winterthur",2023,2023.0,True,OK


In [49]:
# Abweichungen anzeigen
if "BAUJAHR" in df_result.columns:
    df_diff = df_result[df_result["baujahr_gleich"] == False].copy()

    display(df_diff[[
        "adresse_api",
        "BAUJAHR",
        "baujahr_gwr",
        "status_api"
    ]])

,adresse_api,BAUJAHR,baujahr_gwr,status_api
374,"Adlerstrasse 2, 8400 Winterthur",1992,1967.0,OK
815,"Depotplatz 8, 8400 Winterthur",0,NaN,OK
816,"Depotplatz 7, 8400 Winterthur",0,NaN,OK
817,"Depotplatz 6c, 8400 Winterthur",0,NaN,OK
818,"Depotplatz 5, 8400 Winterthur",0,NaN,OK
819,"Depotplatz 3, 8400 Winterthur",0,NaN,OK
820,"Depotplatz 2, 8400 Winterthur",0,NaN,OK
821,"Depotplatz 4c, 8400 Winterthur",0,NaN,OK
822,"Depotplatz 4b, 8400 Winterthur",0,NaN,OK
823,"Depotplatz 4a, 8400 Winterthur",0,NaN,OK


In [50]:
# Nur Zeilen behalten, bei denen BAUJAHR und baujahr_gwr übereinstimmen

if "BAUJAHR" in df_result.columns:
    df_result["BAUJAHR"] = pd.to_numeric(df_result["BAUJAHR"], errors="coerce")
    df_result["baujahr_gwr"] = pd.to_numeric(df_result["baujahr_gwr"], errors="coerce")

    df_result["baujahr_gleich"] = df_result["BAUJAHR"] == df_result["baujahr_gwr"]

    df_ohne_abweichungen = df_result[df_result["baujahr_gleich"] == True].copy()

    print("Anzahl Zeilen vorher:", len(df_result))
    print("Anzahl Zeilen nach Entfernen der Abweichungen:", len(df_ohne_abweichungen))

    display(df_ohne_abweichungen.head())

Anzahl Zeilen vorher: 846
Anzahl Zeilen nach Entfernen der Abweichungen: 814


,EGID,GEB_GEBID,GSW_STATUS,STRASSENNAME,HAUSNR,HAUSNRZUSATZ,PLZ4,ORT,STADTKREIS,HAUPTNUTZUNG,...,Unnamed: 54,adresse_api,feature_id,egid_gwr,baujahr_gwr,label_api,x_api,y_api,status_api,baujahr_gleich
0,210294075,35999,bestehend,Albert-Einstein-Strasse,1,0,8404,Winterthur,Oberwinterthur,Industrie und Gerwerbe,...,NaN,"Albert-Einstein-Strasse 1, 8404 Winterthur",210294075_0,210294075,2024.0,Albert-Einstein-Strasse 1 <b>8404 Winterthur</b>,1263647.750,2700009.75,OK,True
1,191993323,36666,bestehend,Schützenwiesenweg,8,0,8400,Winterthur,Winterthur-Stadt,Verwaltungsgebäude und Gebäude mit öffentliche...,...,NaN,"Schützenwiesenweg 8, 8400 Winterthur",191993323_0,191993323,2024.0,Schützenwiesenweg 8 <b>8400 Winterthur</b>,1261888.125,2696149.25,OK,True
2,191975144,34926,im Bau,Sporrerpark,2,0,8408,Winterthur,Wülflingen,Nebengebäude und dev.Gebäude,...,NaN,"Sporrerpark 2, 8408 Winterthur",191975144_0,191975144,2024.0,Sporrerpark 2 <b>8408 Winterthur</b>,1263772.625,2694387.25,OK,True
3,191975148,34924,im Bau,Sporrerpark,5,0,8408,Winterthur,Wülflingen,Wohngebäude,...,NaN,"Sporrerpark 5, 8408 Winterthur",191975148_1,191975148,2024.0,Sporrerpark 5 <b>8408 Winterthur</b>,1263791.000,2694430.75,OK,True
4,191975148,34924,im Bau,Sporrerpark,4,0,8408,Winterthur,Wülflingen,Wohngebäude,...,NaN,"Sporrerpark 4, 8408 Winterthur",191975148_0,191975148,2024.0,Sporrerpark 4 <b>8408 Winterthur</b>,1263792.375,2694416.25,OK,True


In [51]:
# Export der bereinigten Datei nach Excel
output_path = data_dir / "02_API_Baujahr_Adressen.xlsx"
df_ohne_abweichungen.to_excel(output_path, index=False)

print(f"Gespeichert: {output_path}")

Gespeichert: Data\02_API_Baujahr_Adressen.xlsx
